# Inputs — explorando os workloads

Resumo e forma dos três workloads de `inputs/`: capacidade, mix de eventos,
demanda total de `ALOC` frente à memória disponível e distribuição dos tamanhos.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from simulator.parser import parse_input

WORKLOADS = (
    "inputs/workload_simples.json",
    "inputs/workload.json",
    "inputs/workload_fragmentacao.json",
)

In [ ]:
def resumo(path):
    w = parse_input(path)
    alocs = [e for e in w.events if e.kind == "ALOC"]
    liberas = [e for e in w.events if e.kind == "LIBERA"]
    demanda = sum(e.size for e in alocs)
    return {
        "memoria_total": w.total_memory,
        "eventos": len(w.events),
        "ALOC": len(alocs),
        "LIBERA": len(liberas),
        "demanda_ALOC": demanda,
        "demanda/capacidade": round(demanda / w.total_memory, 2),
        "maior_ALOC": max((e.size for e in alocs), default=0),
        "menor_ALOC": min((e.size for e in alocs), default=0),
    }


for path in WORKLOADS:
    print(path)
    for k, v in resumo(path).items():
        print(f"  {k}: {v}")
    print()

In [ ]:
fig, axes = plt.subplots(1, len(WORKLOADS), figsize=(4 * len(WORKLOADS), 3.2), sharey=True)
for ax, path in zip(axes, WORKLOADS):
    w = parse_input(path)
    sizes = [e.size for e in w.events if e.kind == "ALOC"]
    ax.bar(range(1, len(sizes) + 1), sizes, color="#4a86e8")
    ax.axhline(w.total_memory, color="red", linestyle="--", linewidth=1, label="memória total")
    ax.set_title(Path(path).stem)
    ax.set_xlabel("ALOC #")
    ax.legend(fontsize=8)
axes[0].set_ylabel("tamanho")
fig.tight_layout()
plt.show()